# Final Project: Natural Disaster Severity Prediction

Predict drought severity levels (0-5) for 5 future weeks per region from 91 days of historical meteorological data.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error

import model.utils as utils
import model.train as train

sns.set_style("whitegrid")
%matplotlib inline

## 1. Load Data

In [ ]:
X_train, y_train, train_regions = utils.load_train_data("data/train.csv")
X_test, test_regions = utils.load_test_data("data/test.csv")

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Labels: {y_train.shape}")
print(f"Test:  {X_test.shape[0]} samples, {X_test.shape[1]} features")

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_scores = y_train.flatten()
axes[0].hist(all_scores, bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("Score Distribution (All Samples)")
axes[0].set_xlabel("Severity Score")
axes[0].set_ylabel("Count")

week_means = y_train.mean(axis=0)
axes[1].bar(range(1, 6), week_means, color=sns.color_palette("Set2", 5))
axes[1].set_title("Mean Score per Forecast Week")
axes[1].set_xlabel("Week Ahead")
axes[1].set_ylabel("Mean Score")

plt.tight_layout()
plt.show()

print("Score statistics:")
print(pd.DataFrame(all_scores).describe())

In [ ]:
n_feats = min(30, X_train.shape[1])
top_feats = X_train.iloc[:, :n_feats]

fig, ax = plt.subplots(figsize=(16, 14))
corr = top_feats.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.6})
ax.set_title("Feature Correlation Matrix (First 30 Features)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Baseline Model

XGBoost multi-output regression with default parameters and 5-fold CV.

In [ ]:
scores, mean_mae, std_mae = train.cv_evaluate(X_train, y_train, train_regions, n_splits=5)

for i, s in enumerate(scores, 1):
    print(f"Fold {i}: MAE = {s:.4f}")
print(f"\nMean MAE: {mean_mae:.4f}  (+/- {std_mae:.4f})")

## 4. Predict on Test & Submit

In [ ]:
model = train.train_xgboost(X_train, y_train)
test_preds = model.predict(X_test)
print(f"Prediction shape: {test_preds.shape}")

utils.generate_submission(test_regions, test_preds, "output/submission_xgb.csv")